# Notebook 01 — Generate Distillation Data
**Member: A** | Generates D1 (all traces) and D2 (filtered traces) via Groq API.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os, sys
from google.colab import userdata

# Get Hugging Face token from Colab Secrets
hf_token = userdata.get("HF_TOKEN")

os.environ["HF_TOKEN"] = hf_token
os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token

# Clone GitHub repository
repo_url = "https://github.com/TIFFMUN/SC4001-Group133.git"

if not os.path.exists('/content/SC4001-Group133/.git'):
    os.system(f'git clone {repo_url} /content/SC4001-Group133')

os.chdir('/content/SC4001-Group133')
os.system(f'git pull {repo_url} main')

!pip install -r requirements.txt -q
!pip install -e . -q

  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for sc4001-group133 (pyproject.toml) ... done


In [4]:
# Redirect HF cache + data/ + results/ to Drive — run before any HF imports
import importlib
importlib.invalidate_caches()
from src.setup_cache import setup
setup()

[setup_cache] Cache redirected to Drive: /content/drive/MyDrive/SC4001


In [5]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")
print("Groq API key loaded successfully!")

Groq API key loaded successfully!


In [6]:
# Generate raw CoT traces — resumable if session dies
# Set max_samples: 10 in configs/distillation.yaml for a quick test first
from src.distillation.generate_traces import generate_traces
generate_traces()

Generating traces: 100%|██████████| 100/100 [39:03<00:00, 23.44s/it]


In [7]:
# Filter to correct-answer traces only (D2) — logs filter rate for the report
from src.distillation.filter_traces import filter_traces
filter_traces()

Filtering traces: 100%|██████████| 100/100 [00:00<00:00, 7456.67it/s]


In [8]:
from src.distillation.build_dataset import build_dataset
build_dataset(use_filtered=False)  # D1
build_dataset(use_filtered=True)   # D2

Saving the dataset (0/1 shards):   0%|          | 0/90 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/10 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/77 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/9 [00:00<?, ? examples/s]

In [9]:
# Verify data saved to Drive
import os
for root, _, files in os.walk('/content/drive/MyDrive/SC4001/data'):
    for f in files:
        path = os.path.join(root, f)
        print(f'{path}  ({os.path.getsize(path)/1e6:.1f} MB)')

/content/drive/MyDrive/SC4001/data/README.md  (0.0 MB)
/content/drive/MyDrive/SC4001/data/traces/raw_traces.jsonl  (0.2 MB)
/content/drive/MyDrive/SC4001/data/traces/filtered_traces.jsonl  (0.2 MB)
/content/drive/MyDrive/SC4001/data/processed/all/dataset_dict.json  (0.0 MB)
/content/drive/MyDrive/SC4001/data/processed/all/train/data-00000-of-00001.arrow  (0.1 MB)
/content/drive/MyDrive/SC4001/data/processed/all/train/state.json  (0.0 MB)
/content/drive/MyDrive/SC4001/data/processed/all/train/dataset_info.json  (0.0 MB)
/content/drive/MyDrive/SC4001/data/processed/all/validation/data-00000-of-00001.arrow  (0.0 MB)
/content/drive/MyDrive/SC4001/data/processed/all/validation/state.json  (0.0 MB)
/content/drive/MyDrive/SC4001/data/processed/all/validation/dataset_info.json  (0.0 MB)
/content/drive/MyDrive/SC4001/data/processed/filtered/dataset_dict.json  (0.0 MB)
/content/drive/MyDrive/SC4001/data/processed/filtered/train/data-00000-of-00001.arrow  (0.1 MB)
/content/drive/MyDrive/SC4001/da